## Action Responder Prompt

This notebook hopes to:

- Gather trace data for action-responder agent
- Structure and save as mlflow dataset
- Utilize judges to evaluate current agent
- Try with better model to confirm judges are working
- Run GEPA optimzation to improve

In [1]:
import mlflow

from summit_sim.settings import settings

mlflow.tracing.enable_notebook_display()

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment(settings.mlflow_experiment_name)
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)

In [2]:
# Get all traces from last 7 days (adjust as needed)
# filter_string = """
# tags.graph_type = 'simulation' AND
# tags.agent_name = 'action-responder'
# """
# traces = mlflow.search_traces(
#     locations=[experiment.experiment_id],
#     filter_string=filter_string,
#     max_results=500,
# )

In [3]:
# import json

# spans = [
#     span
#     for _, trace in traces.iterrows()
#     for span in trace['spans']
#     if span["name"] == "action_response_agent"
# ]

# records = [
#     {
#         "inputs": json.loads(span['attributes']['mlflow.spanInputs'])['input_data'],
#         "outputs": json.loads(span['attributes']['mlflow.spanOutputs'])
#     } for span in spans
# ]

# dataset = mlflow.genai.create_dataset(
#     name="action_response_agent",
#     experiment_id=[experiment.experiment_id],
# )
# dataset.merge_records(records)

In [4]:
# Register prompt


SYSTEM_PROMPT = """\
You are an expert Wilderness First Responder (WFR) instructor guiding students through realistic wilderness emergency scenarios.
Your goal is to help students learn proper assessment and treatment protocols while maintaining an engaging, supportive learning environment.

=== YOUR TASK ===
1. Review conversation history to see what has already been discovered
2. Based on the student's action, determine what NEW information to reveal
3. Write narrative_text that describes discoveries naturally
4. Provide encouraging feedback and update completion_score (DO NOT POSE A QUESTION)
5. End narrative with an open question inviting the next action


=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation.
Consider ALL previous actions in the transcript when determining the score. Students build toward milestones across multiple turns.

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

=== BUNDLING & EFFICIENCY ===

If a student completes multiple assessment steps in one action, this is EFFICIENT and should be rewarded with the highest milestone they've reached.
Example: 'I check the scene, approach the patient, and assess airway, breathing, and circulation' = 0.40 (they've done scene + primary + started secondary with vitals mentioned).

=== COMPLETION THRESHOLD ===

completion_score > 0.80 (80%) is sufficient for scenario completion. Students do NOT need a perfect 1.0 to pass. This allows for natural variation in how scenarios unfold.

=== FEEDBACK STYLE - TEACHING + REALISTIC ===

Use encouraging, constructive feedback that acknowledges what the student did right, then gently guides them forward:

Format:
1. Acknowledge their actions: 'Good work! You've completed [what they did].'
2. Describe findings realistically: 'You notice [clinical findings].'
3. Provide a hint about next steps: 'Consider what you'd like to check next' or 'What concerns you most about this patient?'

AVOID:
- 'STOP' language or harsh corrections
- Implying they did something wrong when they assessed properly
- Posing a question in the feedback field

Example feedback:
'Good work! You've completed the scene size-up and primary assessment. You notice the patient has rapid breathing and weak radial pulses.
Your head-to-toe exam reveals puncture wounds consistent with a snakebite. Before proceeding with treatment, consider gathering any additional information you might need.'


=== SCORING RULES ===

1. CUMULATIVE SCORING: Consider ALL previous actions in the transcript. If Turn 1 was scene safety and Turn 2 completes primary assessment, they've earned 0.40.

2. NEVER DECREASE: completion_score must always be >= previous_score

3. BUNDLE REWARD: Multiple steps in one action = jump to highest milestone

=== NARRATIVE & STATE EVOLUTION ===

NARRATIVE_TEXT (2-4 sentences):
- Describe what happens based on student actions
- Progressively reveal hidden information as student performs assessments
- Show realistic patient responses and environmental changes
- If was_correct=False: show consequences
- If was_correct=True: show stabilization or appropriate findings
- End with an open question inviting the next action
- Do NOT repeat information already revealed in conversation history

=== CONTINUITY ===

- Reference previous actions and their effects
- Maintain consistency with established facts
- Show natural progression toward resolution



Guidelines:
- CUMULATIVE SCORING: Consider all previous actions (not just current)
- BE LENIENT: Only flag was_correct=False for explicit treatment without
  assessment (splint, bandage, medication). Assessment is always good.
- BUNDLE REWARD: Multiple steps in one action = jump to highest milestone
- CLOSE ENOUGH = FULL CREDIT: Partial completion counts
- 70% PASS THRESHOLD: Score >= 0.70 completes scenario
- NEVER DECREASE: New score must be >= previous_score
- PROGRESSIVE REVELATION: Only reveal what's discovered this turn,
  don't repeat previously known facts
"""  # noqa: E501

USER_PROMPT_TEMPLATE = """\
Evaluate the following student action in the wilderness rescue scenario.

=== SCENARIO CONTEXT ===
Title: {{title}}
Setting: {{setting}}
Patient Summary: {{patient_summary}}
Hidden Truth: {{hidden_truth}}
Learning Objectives: {{learning_objectives}}

=== GROUND TRUTH (AI Reference Only - Reveal Progressively) ===
Complete medical information to reveal based on student actions:
{{hidden_state}}

=== CONVERSATION HISTORY ===
{% if transcript %}
{% for entry in transcript[-5:] %}
Student: {{ entry.student_action }}
{% if entry.turn_narrative %}
AI: {{ entry.turn_narrative }}
{% endif %}

{% endfor %}
{% else %}
Initial turn - no previous actions.
{% endif %}

=== CURRENT TURN ===
Turn {{turn_count}} of {{max_turns}}
Previous completion_score: {{previous_score}}

Student: {{student_action}}

Generate the response following the narrative_text examples in the schema.
"""


# Register system prompt (every time, no checks)
mlflow.genai.register_prompt(
    name="action-responder-system",
    template=SYSTEM_PROMPT,
)

# Register user prompt (every time, no checks)
mlflow.genai.register_prompt(
    name="action-responder-user",
    template=USER_PROMPT_TEMPLATE,
)

2026/03/30 17:04:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: action-responder-system, version 25
2026/03/30 17:04:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: action-responder-user, version 25


PromptVersion(name=action-responder-user, version=25, template=Evaluate the following student...)

In [5]:
from mlflow.genai.judges import make_judge

JUDGE_MODEL_ENDPOINT = "openrouter:/moonshotai/kimi-k2-thinking"

COMPLETION_JUDGE_INSTRUCTIONS = """\
You are evaluating the scoring accuracy and pedagogical quality of an
AI-generated response in a wilderness first responder (WFR) training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

Only evaluate the final AI agent output below:
{{ outputs }}


TIP:
- Check the transcript to determine what actions have already been taken.


Evaluate against these criteria:
1. Does completion_score align with the WFR rubric based on cumulative actions?
2. Is the completion_score increase from previous turn reasonable (<=0.2 unless explicit bundling)?
3. Does the completion_score always build on itself?
4. Is the agent being overtly strict and not giving an increased completion_score when a new action is taken?
5. If there was no action from the student, the completion_score should not increase from it's previous turn.

Provide a score of 0-1 on how well the AI Agent performs when predicting the completion_score.
1 being the agent did perfect; 0 implying the agent got everything wrong.

Your response should strictly follow this format:
{ "result": float, "rationale": str }
"""


completion_judge = make_judge(
    name="completion-judge",
    instructions=COMPLETION_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [6]:
FEEDBACK_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

Only evaluate the final AI agent output below:
{{ outputs }}


TIP:
- Check the transcript to determine what actions have already been taken.

Evaluate against these criteria:
1. Does feedback contain NO questions and only reflect on historical interactions?
2. Is the feedback personalized to the students response?
3. Is the feedback encouraging and constructive without harsh corrections and focused
    on feedback alone?
4. Is feedback length between 2-4 sentences?

Provide a score of 0-1 on how well the AI Agent performs when generating the feedback.
1 being the agent did perfect; 0 implying the agent got everything wrong.

Your response should strictly follow this format:
{ "result": float, "rationale": str }
"""


feedback_judge = make_judge(
    name="feedback-judge",
    instructions=FEEDBACK_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [7]:
NARRATIVE_JUDGE_INSTRUCTIONS = """\
You are evaluating the structure and quality of an AI-generated response \
in a wilderness first responder training simulation.

AI Input:
{{ inputs }}

Only evaluate the final AI agent output below:
{{ outputs }}


TIP:
- Check the transcript to determine what actions have already been taken.


Evaluate against these criteria:
1. Does narrative_text end with an open question and reveal relevant details?
2. Is the narrative personalized to the students action?
3. Is narrative length between 2-4 sentences?
4. If the student indicated some action like 'check vitals', was some information revealed like 'heart rate 120'?
5. Is any revealed information completely unrelated to the students action?

Provide a score of 0-1 on how well the AI Agent performs when generating the feedback.
1 being the agent did perfect; 0 implying the agent got everything wrong.

Your response should strictly follow this format:
{ "result": float, "rationale": str }
"""


narrative_judge = make_judge(
    name="narrative-judge",
    instructions=NARRATIVE_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [8]:
MEDICAL_JUDGE_INSTRUCTIONS = """\
You are evaluating the medical accuracy of an AI-generated response in a \
wilderness first responder training simulation.

=== PATIENT ASSESSMENT SYSTEM (PAS) - GUIDELINES ===

The PAS follows this general order, but students may bundle steps efficiently or adapt based on the situation:

1. SCENE SIZE-UP: 0-.2 points
   - Check for hazards (environmental dangers, unstable terrain, etc.)
   - Identify mechanism of injury (MOI)
   - Count patients and assess available resources

2. PRIMARY ASSESSMENT (ABCDE): 0-.2 points
   - A: Airway assessment
   - B: Breathing evaluation
   - C: Circulation (pulse, bleeding, shock signs)
   - D: Disability (mental status, AVPU)
   - E: Exposure/Environment (clothing, temperature, elements)

3. SECONDARY ASSESSMENT: 0-.2 points
   - Vital signs (HR, RR, BP, SCTM, temperature)
   - Head-to-Toe exam (systematic physical check)
   - SAMPLE history (Signs/Symptoms, Allergies, Medications, Past history, Last intake, Events leading to injury)

4. TREATMENT: 0-.2 points
   - Address immediate life threats
   - Immobilize injuries
   - Wound care
   - Pain management

5. EVACUATION PLAN: 0-.2 points
   - Stay vs. Go decision
   - Resource planning
   - Timeline establishment

AI Input:
{{ inputs }}

Only evaluate the final AI agent output below:
{{ outputs }}


TIP:
- Reference hidden_truth and hidden_state to determine if treatment was premature.
- Check the transcript to determine what actions have already been taken.

Evaluate against these criteria:
1. Is was_correct accurate?
 - was_correct should be FALSE if student made a medical inacurate action or didn't follow PAS
 - was_correct should be TRUE for correct medical actions aligned to PAS
2. Does was_correct accurately gage the quality of the student's action?
3. Is there anything in the AI response that is not medically accurate?


Provide a score of 0-1 on how well the AI Agent performs when generating the feedback.
1 being the agent did perfect; 0 implying the agent got everything wrong.

Your response should strictly follow this format:
{ "result": float, "rationale": str }
"""


medical_judge = make_judge(
    name="medical-judge",
    instructions=MEDICAL_JUDGE_INSTRUCTIONS,
    model=JUDGE_MODEL_ENDPOINT,
    feedback_value_type=float,
)

In [9]:
scorers = [completion_judge, feedback_judge, narrative_judge, medical_judge]
dataset = mlflow.genai.get_dataset(dataset_id="d-0a54b2d8b67a49bb9ebc4e65b925f5f6")

# results = mlflow.genai.evaluate(
#     data=dataset,
#     scorers=scorers,
# )

In [10]:
from pydantic_ai import Agent
from pydantic_ai.models.openrouter import (
    OpenRouterModel,
    OpenRouterModelSettings,
    OpenRouterProvider,
)

from summit_sim.agents.action_responder import ActionRequest, ActionResponse


@mlflow.trace(span_type="AGENT")
def optimize_wrapper(**kwargs) -> dict:  # noqa: ANN003
    """Parse dict into pydantic request to pass to agent."""
    # Pack MLflow's individual kwargs into your Pydantic model
    input_data = ActionRequest(**kwargs)

    # Load prompts
    system_prompt = mlflow.genai.load_prompt("prompts:/action-responder-system@latest")
    user_prompt = mlflow.genai.load_prompt("prompts:/action-responder-user@latest")

    # Create agent
    provider = OpenRouterProvider(api_key=settings.openrouter_api_key)
    agent = Agent(
        OpenRouterModel(settings.default_model, provider=provider),
        output_type=ActionResponse,
        system_prompt=system_prompt.template,
        model_settings=OpenRouterModelSettings(
            openrouter_reasoning={"effort": "medium"},
            openrouter_usage={"include": True},
        ),
    )

    # Format and run
    prompt = user_prompt.format(
        title=input_data.scenario_title,
        setting=input_data.scenario_setting,
        patient_summary=input_data.patient_summary,
        hidden_truth=input_data.hidden_truth,
        learning_objectives=", ".join(input_data.learning_objectives),
        hidden_state=input_data.hidden_state,
        transcript=input_data.transcript,
        student_action=input_data.student_action,
        turn_count=input_data.turn_count,
        max_turns=input_data.max_turns,
        previous_score=input_data.previous_score,
    )

    result = agent.run_sync(prompt)
    return result.output.model_dump()

In [11]:
import os

from mlflow.genai.optimize.optimizers import GepaPromptOptimizer

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"] = "True"

REFLECTION_MODEL = "openrouter:/moonshotai/kimi-k2-thinking"

result = mlflow.genai.optimize_prompts(
    predict_fn=optimize_wrapper,
    train_data=dataset,
    prompt_uris=[
        "prompts:/action-responder-system@latest",
        "prompts:/action-responder-user@latest",
    ],
    optimizer=GepaPromptOptimizer(
        reflection_model=REFLECTION_MODEL,
        display_progress_bar=True,
        max_metric_calls=50,
    ),
    # optimizer=MetaPromptOptimizer(
    #     reflection_model=REFLECTION_MODEL,
    #     guidelines="This prompt is for a Wilderness First Responder agent which dynamically responds to students actions in an emergency situation",
    # ),
    scorers=scorers,
)

/home/bhamm/repos/summit-sim/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
GEPA Optimization:   0%|          | 0/50 [00:00<?, ?rollouts/s]2026/03/30 17:04:24 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
17:04:28 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:04:28 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:04:28 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openroute

Iteration 0: Base program full valset score: 0.7471153846153846 over 13 / 13 examples
Iteration 1: Selected program 0 score: 0.7471153846153846


17:13:36 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:13:36 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:13:36 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:13:36 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:13:36 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:13:36 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:13:44 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
2026-03-30 17:13:44 [INFO] LiteLLM: Wrapper: Completed Call, calling success_handler
17:13:44 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = open


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



17:18:47 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
2026-03-30 17:18:47 [INFO] LiteLLM: Wrapper: Completed Call, calling success_handler


...

17:56:58 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:56:58 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:56:58 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:56:58 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:56:58 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
2026-03-30 17:56:58 [INFO] LiteLLM: 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = openrouter
17:57:45 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
2026-03-30 17:57:45 [INFO] LiteLLM: Wrapper: Completed Call, calling success_handler
17:57:45 - LiteLLM:INFO: utils.py:3995 - 
LiteLLM completion() model= moonshotai/kimi-k2-thinking; provider = open

Iteration 4: New subsample score 0.0 is not better than old score 2.6625, skipping


2026/03/30 17:58:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: action-responder-user, version 26


🏃 View run flawless-mink-710 at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/9bac579f89f04697b5048096a71975d5
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-53f7dcb42342958adaded8b50bf75dd0), Trace(trace_id=tr-9b5819f755a891824520e68ba9192d2a), Trace(trace_id=tr-48c6d65d645edb1185b45b7f57a0e107), Trace(trace_id=tr-3bf71eeee0a81cf2c587b58154478da3), Trace(trace_id=tr-91553332708cadc3395bc62ed9669250), Trace(trace_id=tr-633c72538f9b196f237d94274673470d), Trace(trace_id=tr-e65da8aaa9d68aa267891fbeca54aea9), Trace(trace_id=tr-11e18e764312c3420f3e4853d26442cf), Trace(trace_id=tr-ea20509e59a30cb0637cff9df36e79b7), Trace(trace_id=tr-7e94f3e78c6a20aff0d42ff154913224)]

In [12]:
models_to_test = [
    "google/gemini-3.1-flash-lite-preview",
    "qwen/qwen3.5-flash-02-23",
    "qwen/qwen3-235b-a22b-2507",
    "openai/gpt-4o-mini",
    "openai/gpt-4.1-mini",
    "openai/gpt-5-nano",
    "openai/gpt-5.4-nano",
    "openai/gpt-oss-120b",
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
    "xiaomi/mimo-v2-flash",
    "z-ai/glm-4.7-flash",
    "",
]